# Hi-cGAN Faz 1: Temiz RNA-RNA cGAN (`rna-cgan-clean`) — Google Colab Eğitim & Değerlendirme

Bu defter, `rna-cgan-clean` dalındaki temiz **Pix2Pix cGAN** modelini Google Colab GPU üzerinde eğitmek, `splits.txt` test kromozomları üzerinde tam kromozom tahminleri üretmek ve ortak hakem (`hicgan/lib/evaluation.py`) ile skorlamak için hazırlanmıştır.

**Ortak Deney Sözleşmesi (Faz 1):**
- **Koşullandırma:** `SplitChromatinStreamsLayer` + Siamese Dilated `Conv1D` + `Pairwise2DEmbeddingLayer` (`4` mesafe kanalı + opsiyonel 2B `kmer7`)
- **Özellikler:** `feature_pack` (`representative`: 24+3=27 1B özellik + `kmer7` 2B çift özelliği)
- **Hedef Ölçeği:** Yalnızca train kromozomlarından küresel `P99` kırpmalı `ln(1 + 10x)` (`target_value_range.json`), çıkarımda ham sayım `.npz` formatına geri dönüşüm
- **Pencereleme:** `splits.txt` (`12` Train / `3` Val / `8` Test), üst üçgen `grid` (`windowSize=64`, `gridStride=16`, `flankingSize=64`, `flankPad='all'`) + **SAT boş pencere filtresi**
- **Eğitim Hijyeni:** `CosineDecay(alpha=0.05)`, deterministik `validationStep(training=False)`, `EarlyStopping(patience=10)` (`val_pixel_loss`), `Dropout=0.0`, `ReduceOp.MEAN`


## 1. GPU Kontrolü ve Google Drive Bağlantısı


In [ ]:
import os
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("Algılanan GPU:", gpus)
!nvidia-smi || true

USE_DRIVE = True  # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## 2. Depoyu Klonlama / `rna-cgan-clean` Dalına Geçme
> **Not:** `rna-cgan-clean` dalını henüz GitHub'a pushlamadıysanız önce yerel makinenizde `git push -u origin rna-cgan-clean` çalıştırın (veya zip/Drive üzerinden `/content/Hi-cGAN` içine yükleyin).


In [ ]:
REPO_URL = "https://github.com/nehirvardar/Hi-cGAN.git"  # @param {type:"string"}
BRANCH = "rna-cgan-clean"  # @param {type:"string"}
REPO_DIR = "/content/Hi-cGAN"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd /content/Hi-cGAN
!git log -n 3 --oneline


## 3. Bağımlılıkları Kurma ve İsteğe Bağlı Duman Testi


In [ ]:
!pip install -q openpyxl cooler seaborn scikit-image tqdm
!pip install -q -e . --no-deps

RUN_UNIT_TESTS = False  # @param {type:"boolean"}
if RUN_UNIT_TESTS:
    !python -m unittest discover -s tests -v


## 4. Eğitim Parametreleri ve Veri Doğrulama


In [ ]:
DATA_DIR = "hicgan/data/rna_gt"
SPLIT_FILE = "hicgan/data/splits.txt"
CHROMATIN_DIR = "data/feature_pack"

# Özellik ve mimari ayarları (Faz 1 & Faz 2 ortak sözleşme)
FEATURE_SET = "representative"  # @param ["representative", "direct", "tier1_2", "gm12878", "hela", "all"]
USE_PAIR_FEATURES = True        # @param {type:"boolean"}
WINDOW_SIZE = 64                # @param [64, 128, 256] {type:"raw"}
GRID_STRIDE = 16                # @param {type:"integer"}
BIN_SIZE = 250000
TARGET_NORM = "p99_log10"       # @param ["p99_log10", "minmax_log1p", "unit_range", "none"]
BLEND_MODE = "raised_cosine_peak"  # @param ["raised_cosine_peak", "uniform"]

# Eğitim hiperparametreleri
EPOCHS = 50                     # @param {type:"integer"}
BATCH_SIZE = 16                 # @param {type:"integer"}
EARLY_STOP = 10                 # @param {type:"integer"}
DROPOUT = 0.0                   # @param {type:"number"}
FG_LOSS_WEIGHT = 10.0           # @param {type:"number"}
UNDERPRED_BOOST = 1.5           # @param {type:"number"}
LR_GEN = 2e-5                   # @param {type:"number"}
LR_DISC = 1e-6                  # @param {type:"number"}
SEED = 42                       # @param {type:"integer"}
RESUME = False                  # @param {type:"boolean"}

RUN_NAME = f"rna_cgan_clean_{FEATURE_SET}_ws{WINDOW_SIZE}_gs{GRID_STRIDE}"
BASE_OUT = "/content/drive/MyDrive/hicgan_runs" if USE_DRIVE else "/content/hicgan_runs"
OUTPUT_DIR = os.path.join(BASE_OUT, RUN_NAME)
PRED_DIR = os.path.join(OUTPUT_DIR, "test_eval")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for p in (DATA_DIR, SPLIT_FILE, CHROMATIN_DIR):
    assert os.path.exists(p), f"Eksik yol: {p}"
print("Çıktı klasörü:", OUTPUT_DIR)
!cat {SPLIT_FILE}


## 5. Eğitimi Başlatma (`hicgan.training`)


In [ ]:
pair_flag = "--usePairFeatures" if USE_PAIR_FEATURES else "--noPairFeatures"
resume_flag = "--resume" if RESUME else ""

cmd = (
    f"python -m hicgan.training "
    f"--dataDir {DATA_DIR} "
    f"--splitFile {SPLIT_FILE} "
    f"--trainingChromatinFolders {CHROMATIN_DIR} "
    f"--validationChromatinFolders {CHROMATIN_DIR} "
    f"--inputMode feature_pack "
    f"--featureSet {FEATURE_SET} "
    f"{pair_flag} "
    f"--windowMode grid "
    f"--windowSize {WINDOW_SIZE} "
    f"--gridStride {GRID_STRIDE} "
    f"--binSize {BIN_SIZE} "
    f"--targetNorm {TARGET_NORM} "
    f"--epochs {EPOCHS} "
    f"--batchSize {BATCH_SIZE} "
    f"--earlyStop {EARLY_STOP} "
    f"--dropout {DROPOUT} "
    f"--foregroundLossWeight {FG_LOSS_WEIGHT} "
    f"--underpredBoost {UNDERPRED_BOOST} "
    f"--learningRateGenerator {LR_GEN} "
    f"--learningRateDiscriminator {LR_DISC} "
    f"--plotFrequency 5 "
    f"--keepTFRecords "
    f"--seed {SEED} "
    f"{resume_flag} "
    f'--outputFolder "{OUTPUT_DIR}"'
)
print("Çalıştırılacak komut:\n", cmd)
!{cmd}


## 6. Eğitim Kayıp Eğrisi ve Örnek Çıktılar


In [ ]:
import glob
from IPython.display import display, Image

loss_png = os.path.join(OUTPUT_DIR, "lossOverEpochs.png")
if os.path.exists(loss_png):
    display(Image(filename=loss_png))

sample_pngs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "testpred_epoch_*.png")))
if sample_pngs:
    print("Son örnek tahmin:", sample_pngs[-1])
    display(Image(filename=sample_pngs[-1]))


## 7. Test Kromozomlarında Tam Tahmin ve Ortak `evaluation.py` Skorlaması


In [ ]:
best_model = os.path.join(OUTPUT_DIR, "generator_best.keras")
if not os.path.exists(best_model):
    best_model = os.path.join(OUTPUT_DIR, "generator_final.keras")
assert os.path.exists(best_model), f"Eğitilmiş model bulunamadı: {best_model}"

pred_cmd = (
    f"python -m hicgan.predict "
    f'--trainedModel "{best_model}" '
    f"--dataDir {DATA_DIR} "
    f"--splitFile {SPLIT_FILE} "
    f"--split test "
    f"--predictionChromosomesFolders {CHROMATIN_DIR} "
    f"--inputMode feature_pack "
    f"--windowMode grid "
    f"--windowSize {WINDOW_SIZE} "
    f"--gridStride {GRID_STRIDE} "
    f"--binSize {BIN_SIZE} "
    f"--blendMode {BLEND_MODE} "
    f"--batchSize {BATCH_SIZE} "
    f"--runEval "
    f"--gtDir {DATA_DIR} "
    f'--outputFolder "{PRED_DIR}"'
)
print("Tahmin + Değerlendirme komutu:\n", pred_cmd)
!{pred_cmd}


## 8. Çekirdek Karne (`core_metrics_scorecard.csv`) ve Tanı Grafikleri


In [ ]:
import pandas as pd

scorecard_path = os.path.join(PRED_DIR, "evaluation", "core_metrics_scorecard.csv")
if os.path.exists(scorecard_path):
    df_sc = pd.read_csv(scorecard_path)
    display(df_sc)
else:
    print("Scorecard bulunamadı:", scorecard_path)

eval_pngs = sorted(glob.glob(os.path.join(PRED_DIR, "evaluation", "*.png")))
for p in eval_pngs[:8]:
    print(os.path.basename(p))
    display(Image(filename=p))
